In [ ]:
import fastf1 as ff1
import pandas as pd
import numpy as np
from fastf1 import plotting
from sklearn.preprocessing import StandardScaler
import os
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns


if not os.path.exists('cache'):
    os.makedirs('cache')
ff1.Cache.enable_cache('cache')

In [ ]:
# Carga de Datos
session = ff1.get_session(2025, 'Italian Grand Prix', 'R')
session.load()
print(f"Sesión cargada: {session.event.EventName} {session.name}")

In [ ]:
# Definición de Zonas de Interés
# Se definen zonas de análisis con sub-partes: 'braking', 'apex', 'exit'
circuit_info = session.get_circuit_info()
analysis_zones = {}

if circuit_info is None:
    print("Error: Could not load circuit information. Please check your internet connection or try again later.")
    print("Skipping analysis zone definition due to missing circuit info.")
else:
    print(f"Total corners found: {len(circuit_info.corners)}")
    try:
        if len(circuit_info.corners) > 1:
            analysis_zones['T1_Rettifilo'] = {
                'braking': (circuit_info.corners.loc[0]['Distance'] - 350, circuit_info.corners.loc[0]['Distance']),
                'apex': (circuit_info.corners.loc[0]['Distance'] - 50, circuit_info.corners.loc[1]['Distance'] + 50),
                'exit': (circuit_info.corners.loc[1]['Distance'], circuit_info.corners.loc[1]['Distance'] + 250)
            }
        else:
            print("Not enough corners found to define T1_Rettifilo analysis zone.")

        if len(circuit_info.corners) > 4:
            analysis_zones['T4_Roggia'] = {
                'braking': (circuit_info.corners.loc[3]['Distance'] - 300, circuit_info.corners.loc[3]['Distance']),
                'apex': (circuit_info.corners.loc[3]['Distance'] - 50, circuit_info.corners.loc[4]['Distance'] + 50),
                'exit': (circuit_info.corners.loc[4]['Distance'], circuit_info.corners.loc[4]['Distance'] + 250)
            }
        else:
            print("Not enough corners found to define T4_Roggia analysis zone.")

        if len(circuit_info.corners) > 10:
             analysis_zones['T11_Alboreto'] = {
                'braking': (circuit_info.corners.loc[10]['Distance'] - 350, circuit_info.corners.loc[10]['Distance']),
                'apex': (circuit_info.corners.loc[10]['Distance'] - 100, circuit_info.corners.loc[10]['Distance'] + 100),
                'exit': (circuit_info.corners.loc[10]['Distance'] + 100, circuit_info.corners.loc[10]['Distance'] + 350)
            }
        else:
            print("Not enough corners found to define T11_Alboreto analysis zone.")

    except Exception as e:
        print(f"An error occurred while defining analysis zones: {e}")


print(f"Defined {len(analysis_zones)} analysis zones.")

In [ ]:
# Funciones de Feature Engineering

def get_braking_point(telemetry_fragment):
    """Devuelve la distancia donde 'Brake' se activa."""
    try:
        # Ensure the fragment is not empty and has the 'Brake' column
        if telemetry_fragment.empty or 'Brake' not in telemetry_fragment.columns:
            return np.nan
        braking_start_row = telemetry_fragment[telemetry_fragment['Brake'] > 0.1].first_valid_index()
        if braking_start_row is not None:
            return telemetry_fragment.loc[braking_start_row, 'Distance']
    except Exception as e:
        pass
    return np.nan

def get_min_apex_speed(telemetry_fragment):
    """Devuelve la velocidad mínima en el fragmento de vértice."""
    try:
        if not telemetry_fragment.empty and 'Speed' in telemetry_fragment.columns:
            return telemetry_fragment['Speed'].min()
    except Exception as e:
        pass
    return np.nan

def get_max_deceleration(telemetry_fragment):
    """Devuelve la desaceleración máxima (en m/s^2) mientras se frena."""
    try:
        if telemetry_fragment.empty or 'Brake' not in telemetry_fragment.columns or 'Acceleration' not in telemetry_fragment.columns:
            return np.nan

        braking_fragment = telemetry_fragment[telemetry_fragment['Brake'] > 0.1]
        if not braking_fragment.empty:

            return braking_fragment['Acceleration'].min()
    except Exception as e:
        pass
    return np.nan

def get_max_throttle_rate(telemetry_fragment):
    """Devuelve la tasa máxima de aplicación del acelerador (% por segundo)."""
    try:
        if not telemetry_fragment.empty and 'Throttle' in telemetry_fragment.columns and 'TimeDelta' in telemetry_fragment.columns:

            throttle_rate = telemetry_fragment['Throttle'].diff() / telemetry_fragment['TimeDelta'].replace(0, np.nan)

            max_rate = throttle_rate[throttle_rate > 0].max()

            if pd.notna(max_rate) and not np.isinf(max_rate):
                return max_rate
    except Exception as e:
        pass
    return np.nan


In [ ]:
# Iteración por todos los pilotos y promedio de vueltas

driver_list = session.drivers
print(f"Analizando {len(driver_list)} pilotos...")

all_driver_features_avg = []

for driver_code in driver_list:

    all_laps = session.laps.pick_drivers(driver_code).pick_quicklaps()

    if all_laps.empty:
        print(f"Saltando a {driver_code} (sin vueltas válidas para analizar)")
        continue

    lap_features_list = []

    for lap_index, lap in all_laps.iterlaps():

        try:
            telemetry = lap.get_car_data().add_distance()


            telemetry['Speed_ms'] = telemetry['Speed'] / 3.6  # Convertir km/h a m/s
            telemetry['TimeDelta'] = telemetry['Time'].diff().dt.total_seconds()
            telemetry['SpeedDelta'] = telemetry['Speed_ms'].diff()
            # Aceleración en m/s^2
            telemetry['Acceleration'] = telemetry['SpeedDelta'] / telemetry['TimeDelta']

            if telemetry.empty:

                continue

            lap_data = {}

            for zone_name, zone_parts in analysis_zones.items():

                # Fragmentar la telemetría según las sub-zonas
                braking_fragment = telemetry.loc[
                    (telemetry['Distance'] >= zone_parts['braking'][0]) & (telemetry['Distance'] <= zone_parts['braking'][1])
                ]
                apex_fragment = telemetry.loc[
                    (telemetry['Distance'] >= zone_parts['apex'][0]) & (telemetry['Distance'] <= zone_parts['apex'][1])
                ]
                exit_fragment = telemetry.loc[
                    (telemetry['Distance'] >= zone_parts['exit'][0]) & (telemetry['Distance'] <= zone_parts['exit'][1])
                ]

                # --- 2. Calcular y guardar cada feature de la vuelta ---
                lap_data[f'{zone_name}_BrakingPoint'] = get_braking_point(braking_fragment)
                lap_data[f'{zone_name}_MaxDecel'] = get_max_deceleration(braking_fragment)
                lap_data[f'{zone_name}_ApexSpeed'] = get_min_apex_speed(apex_fragment)
                lap_data[f'{zone_name}_ThrottleRate'] = get_max_throttle_rate(exit_fragment)

            lap_features_list.append(lap_data)

        except Exception as e:
            # print(f"Error procesando vuelta {lap.LapNumber} de {driver_code}: {e}")
            continue


    # --- 3. Calcular el promedio para el piloto ---
    if not lap_features_list: # Si no se pudo calcular features para ninguna vuelta
        print(f"Saltando a {driver_code} (no se pudieron calcular features para ninguna vuelta)")
        continue

    # Convertimos la lista de features de vueltas en un DataFrame
    df_laps_driver = pd.DataFrame(lap_features_list)

    # Calculamos la media de todas las features en todas las vueltas, excluyendo NaNs
    driver_avg_features = df_laps_driver.mean(skipna=True) # Esto nos da una Serie con el promedio

    # Añadimos el código del piloto
    driver_avg_features['Driver'] = driver_code

    # Añadimos esta fila de promedios a nuestra lista maestra
    all_driver_features_avg.append(driver_avg_features)

In [ ]:
# --- 5. Creación y Normalización del DataFrame ---

# Convertimos la lista de Series promediadas en un DataFrame
# Usamos dropna(how='all') para quitar pilotos que solo tienen NaNs
df_raw_features = pd.DataFrame(all_driver_features_avg).set_index('Driver').dropna(how='all')

print("\n--- DataFrame de Features (Datos Promediados) ---")
print(df_raw_features)

if df_raw_features.empty:
    print("No hay datos suficientes para normalizar. Revisa la carga de datos.")
else:
    # Calculamos la media de cada feature (columna)
    feature_averages = df_raw_features.mean()

    # NORMALIZACIÓN: Restamos la media de la feature a cada piloto
    df_normalized = df_raw_features - feature_averages

    print("\n--- DataFrame Normalizado (Puntuación de Estilo Promedio) ---")
    print(df_normalized)
    print("\nInterpretación del DataFrame Normalizado:")
    print("Valores positivos = 'más que el promedio' (ej. frenó más tarde, fue más rápido en vértice, etc.)")
    print("Valores negativos = 'menos que el promedio'")



In [ ]:
# --- 6. ESTANDARIZACIÓN (Paso para Clustering) ---

print("\n--- Estandarizando datos para Clustering ---")

    # Eliminamos columnas con NaN o Inf que puedan causar problemas en StandardScaler
df_normalized_cleaned = df_normalized.replace([np.inf, -np.inf], np.nan).dropna(axis=1)

if df_normalized_cleaned.empty:
        print("No hay columnas válidas para estandarizar después de limpiar NaNs/Infs.")
else:
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(df_normalized_cleaned)

        # Creamos un nuevo DataFrame con los datos estandarizados
        df_standardized = pd.DataFrame(
            data=scaled_data,
            index=df_normalized_cleaned.index,
            columns=df_normalized_cleaned.columns
        )

        print("\n--- DataFrame Estandarizado (Listo para Clustering/PCA/t-SNE) ---")
        display(df_standardized) # Mostrar head para evitar salida excesiva
        print("\nInterpretación del DataFrame Estandarizado:")
        print("Todas las features están ahora en la misma escala (Z-scores, media=0, std=1).")
        print("Este es el input final para tu análisis de machine learning.")
        print("Este es el input final para tu análisis de machine learning.")

In [ ]:
lap = session.laps.pick_driver('43')

# Obtener datos de telemetría y añadir distancia
telemetry = lap.get_car_data().add_distance()


In [ ]:
telemetry

### t-Distributed Stochastic Neighbor Embedding (t-SNE) for Driver Style Analysis

t-SNE is a dimensionality reduction technique used for visualizing high-dimensional data in a low-dimensional space (typically 2D or 3D). It's particularly effective at revealing clusters and patterns in data that might be obscured in higher dimensions.

We will apply t-SNE to our `df_standardized` DataFrame, which contains the standardized driving style features for each driver. This will help us identify groups of drivers with similar driving characteristics based on the features we've engineered.

In [ ]:
# Initialize t-SNE
# perplexity: balance attention between local and global aspects of data
# n_components: desired dimensionality of the output (2D for visualization)
# random_state: for reproducibility
t_sne = TSNE(n_components=2, perplexity=5, random_state=42)

# Fit and transform the standardized data
t_sne_results = t_sne.fit_transform(df_standardized)

# Create a DataFrame for easier plotting
df_tsne = pd.DataFrame(data=t_sne_results, columns=['TSNE1', 'TSNE2'], index=df_standardized.index)

print("t-SNE embedding generated successfully!")
display(df_tsne.head())

In [ ]:
plt.figure(figsize=(12, 10))
sns.scatterplot(
    x='TSNE1', y='TSNE2',
    data=df_tsne,
    hue=df_tsne.index, # Color points by driver
    palette='tab20', # Use a distinct color palette
    s=150, # Size of the points
    alpha=0.8
)

# Add driver labels to the points
for i, driver in enumerate(df_tsne.index):
    plt.annotate(driver, (df_tsne['TSNE1'].iloc[i] + 0.5, df_tsne['TSNE2'].iloc[i]), fontsize=9)

plt.title('t-SNE Visualization of Driver Driving Styles', fontsize=16)
plt.xlabel('t-SNE Component 1', fontsize=12)
plt.ylabel('t-SNE Component 2', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Driver')
plt.tight_layout()
plt.show()

# Task
Create markdown text to explain Principal Component Analysis (PCA) and its application to the standardized driving style features.

## explain_pca

### Subtask:
Add a markdown cell to explain Principal Component Analysis (PCA) and its application to the standardized driving style features.


## Principal Component Analysis (PCA) for Driver Style

### What is Principal Component Analysis (PCA)?
Principal Component Analysis (PCA) is a powerful statistical technique used primarily for **dimensionality reduction** and **feature extraction**. Its main goal is to transform a dataset with many correlated variables into a new set of uncorrelated variables called **Principal Components (PCs)**. The first principal component accounts for the largest possible variance in the data, and each subsequent component accounts for the remaining variance in decreasing order of importance. Essentially, PCA finds the directions (principal components) along which the data varies the most.

### Application to Standardized Driving Style Features (`df_standardized`)
In our analysis, we have generated a `df_standardized` DataFrame containing various standardized driving style features for each driver. This DataFrame is high-dimensional, meaning it has many columns (features) describing different aspects of a driver's style (e.g., braking points, apex speeds, throttle rates across multiple corners).

Applying PCA to `df_standardized` involves the following steps conceptually:
1.  **Identify Principal Components**: PCA will calculate a set of principal components based on the variance and covariance among all the features in `df_standardized`.
2.  **Project Data**: Each driver's data point (a row in `df_standardized`) will be projected onto these principal components. This transforms the original feature space into a new, lower-dimensional space defined by the PCs.
3.  **Select Components**: We can then choose a subset of these principal components (e.g., the first two or three) that capture most of the original variance in the data. This effectively reduces the number of dimensions we need to consider.

### Benefits for Driver Behavior Visualization and Understanding
By reducing the dimensionality of our `df_standardized` DataFrame using PCA, we gain several key advantages:
*   **Visualization**: It becomes possible to visualize complex, high-dimensional driver data in a simple 2D or 3D scatter plot. Each point in this plot would represent a driver, and their position relative to others would indicate similarities or differences in their overall driving style based on the combined features.
*   **Pattern Recognition**: PCA helps in uncovering underlying patterns and relationships within the driver data that might not be obvious in the original high-dimensional space. We can identify groups of drivers with similar characteristics (e.g., aggressive brakers, smooth cornerers) even if those characteristics are combinations of multiple individual features.
*   **Noise Reduction**: By focusing on components that explain the most variance, PCA can also help in filtering out less informative or noisy features, leading to a clearer representation of the core driving styles.

## Principal Component Analysis (PCA) for Driver Style

### What is Principal Component Analysis (PCA)?
Principal Component Analysis (PCA) is a powerful statistical technique used primarily for **dimensionality reduction** and **feature extraction**. Its main goal is to transform a dataset with many correlated variables into a new set of uncorrelated variables called **Principal Components (PCs)**. The first principal component accounts for the largest possible variance in the data, and each subsequent component accounts for the remaining variance in decreasing order of importance. Essentially, PCA finds the directions (principal components) along which the data varies the most.

### Application to Standardized Driving Style Features (`df_standardized`)
In our analysis, we have generated a `df_standardized` DataFrame containing various standardized driving style features for each driver. This DataFrame is high-dimensional, meaning it has many columns (features) describing different aspects of a driver's style (e.g., braking points, apex speeds, throttle rates across multiple corners).

Applying PCA to `df_standardized` involves the following steps conceptually:
1.  **Identify Principal Components**: PCA will calculate a set of principal components based on the variance and covariance among all the features in `df_standardized`.
2.  **Project Data**: Each driver's data point (a row in `df_standardized`) will be projected onto these principal components. This transforms the original feature space into a new, lower-dimensional space defined by the PCs.
3.  **Select Components**: We can then choose a subset of these principal components (e.g., the first two or three) that capture most of the original variance in the data. This effectively reduces the number of dimensions we need to consider.

### Benefits for Driver Behavior Visualization and Understanding
By reducing the dimensionality of our `df_standardized` DataFrame using PCA, we gain several key advantages:
*   **Visualization**: It becomes possible to visualize complex, high-dimensional driver data in a simple 2D or 3D scatter plot. Each point in this plot would represent a driver, and their position relative to others would indicate similarities or differences in their overall driving style based on the combined features.
*   **Pattern Recognition**: PCA helps in uncovering underlying patterns and relationships within the driver data that might not be obvious in the original high-dimensional space. We can identify groups of drivers with similar characteristics (e.g., aggressive brakers, smooth cornerers) even if those characteristics are combinations of multiple individual features.
*   **Noise Reduction**: By focusing on components that explain the most variance, PCA can also help in filtering out less informative or noisy features, leading to a clearer representation of the core driving styles.


## Principal Component Analysis (PCA) for Driver Style

### What is Principal Component Analysis (PCA)?
Principal Component Analysis (PCA) is a powerful statistical technique used primarily for **dimensionality reduction** and **feature extraction**. Its main goal is to transform a dataset with many correlated variables into a new set of uncorrelated variables called **Principal Components (PCs)**. The first principal component accounts for the largest possible variance in the data, and each subsequent component accounts for the remaining variance in decreasing order of importance. Essentially, PCA finds the directions (principal components) along which the data varies the most.

### Application to Standardized Driving Style Features (`df_standardized`)
In our analysis, we have generated a `df_standardized` DataFrame containing various standardized driving style features for each driver. This DataFrame is high-dimensional, meaning it has many columns (features) describing different aspects of a driver's style (e.g., braking points, apex speeds, throttle rates across multiple corners).

Applying PCA to `df_standardized` involves the following steps conceptually:
1.  **Identify Principal Components**: PCA will calculate a set of principal components based on the variance and covariance among all the features in `df_standardized`.
2.  **Project Data**: Each driver's data point (a row in `df_standardized`) will be projected onto these principal components. This transforms the original feature space into a new, lower-dimensional space defined by the PCs.
3.  **Select Components**: We can then choose a subset of these principal components (e.g., the first two or three) that capture most of the original variance in the data. This effectively reduces the number of dimensions we need to consider.

### Benefits for Driver Behavior Visualization and Understanding
By reducing the dimensionality of our `df_standardized` DataFrame using PCA, we gain several key advantages:
*   **Visualization**: It becomes possible to visualize complex, high-dimensional driver data in a simple 2D or 3D scatter plot. Each point in this plot would represent a driver, and their position relative to others would indicate similarities or differences in their overall driving style based on the combined features.
*   **Pattern Recognition**: PCA helps in uncovering underlying patterns and relationships within the driver data that might not be obvious in the original high-dimensional space. We can identify groups of drivers with similar characteristics (e.g., aggressive brakers, smooth cornerers) even if those characteristics are combinations of multiple individual features.
*   **Noise Reduction**: By focusing on components that explain the most variance, PCA can also help in filtering out less informative or noisy features, leading to a clearer representation of the core driving styles.

## Principal Component Analysis (PCA) for Driver Style

### What is Principal Component Analysis (PCA)?
Principal Component Analysis (PCA) is a powerful statistical technique used primarily for **dimensionality reduction** and **feature extraction**. Its main goal is to transform a dataset with many correlated variables into a new set of uncorrelated variables called **Principal Components (PCs)**. The first principal component accounts for the largest possible variance in the data, and each subsequent component accounts for the remaining variance in decreasing order of importance. Essentially, PCA finds the directions (principal components) along which the data varies the most.

### Application to Standardized Driving Style Features (`df_standardized`)
In our analysis, we have generated a `df_standardized` DataFrame containing various standardized driving style features for each driver. This DataFrame is high-dimensional, meaning it has many columns (features) describing different aspects of a driver's style (e.g., braking points, apex speeds, throttle rates across multiple corners).

Applying PCA to `df_standardized` involves the following steps conceptually:
1.  **Identify Principal Components**: PCA will calculate a set of principal components based on the variance and covariance among all the features in `df_standardized`.
2.  **Project Data**: Each driver's data point (a row in `df_standardized`) will be projected onto these principal components. This transforms the original feature space into a new, lower-dimensional space defined by the PCs.
3.  **Select Components**: We can then choose a subset of these principal components (e.g., the first two or three) that capture most of the original variance in the data. This effectively reduces the number of dimensions we need to consider.

### Benefits for Driver Behavior Visualization and Understanding
By reducing the dimensionality of our `df_standardized` DataFrame using PCA, we gain several key advantages:
*   **Visualization**: It becomes possible to visualize complex, high-dimensional driver data in a simple 2D or 3D scatter plot. Each point in this plot would represent a driver, and their position relative to others would indicate similarities or differences in their overall driving style based on the combined features.
*   **Pattern Recognition**: PCA helps in uncovering underlying patterns and relationships within the driver data that might not be obvious in the original high-dimensional space. We can identify groups of drivers with similar characteristics (e.g., aggressive brakers, smooth cornerers) even if those characteristics are combinations of multiple individual features.
*   **Noise Reduction**: By focusing on components that explain the most variance, PCA can also help in filtering out less informative or noisy features, leading to a clearer representation of the core driving styles.


## perform_pca

### Subtask:
Perform PCA on the `df_standardized` DataFrame using `sklearn.decomposition.PCA` and create a DataFrame with the principal components.


**Reasoning**:
The subtask requires performing PCA on the `df_standardized` DataFrame. This code block will import the `PCA` class, instantiate it, fit and transform the data, and then create a new DataFrame `df_pca` with the principal components as specified in the instructions.



In [ ]:
from sklearn.decomposition import PCA

# Initialize PCA with 2 components
pca = PCA(n_components=2, random_state=42)

# Fit and transform the standardized data
pca_results = pca.fit_transform(df_standardized)

# Create a DataFrame for easier plotting with driver codes as index
df_pca = pd.DataFrame(data=pca_results, columns=['PC1', 'PC2'], index=df_standardized.index)

print("PCA performed successfully!")
display(df_pca.head())

## visualize_pca

### Subtask:
Create a scatter plot of the first two principal components, labeling each point with the driver's code to visualize clusters and relationships.


## Summary:

### Data Analysis Key Findings

*   **PCA Explanation**: Principal Component Analysis (PCA) was explained as a technique for dimensionality reduction and feature extraction, transforming correlated variables into uncorrelated Principal Components (PCs), ordered by the variance they explain. For the standardized driving style features (`df_standardized`), PCA aims to identify key components that capture the most variance, project driver data onto these components, and allow for visualization and pattern recognition in a lower-dimensional space.
*   **PCA Application**: PCA was successfully applied to the `df_standardized` DataFrame.
*   **Dimensionality Reduction**: The dimensionality was reduced to two principal components.
*   **PCA Results DataFrame**: A new DataFrame, `df_pca`, was created, containing the first two principal components, labeled 'PC1' and 'PC2', with driver codes retained as the index. For example, the `df_pca.head()` displayed values for drivers like 1, 4, 81, 16, and 63.

### Insights or Next Steps

*   The creation of `df_pca` with two principal components sets the stage for easy visualization of driver clusters and relationships, which was the primary benefit highlighted for applying PCA to high-dimensional driving style data.
*   The next logical step is to visualize these two principal components (PC1 and PC2) in a scatter plot to identify potential groupings or outliers among drivers based on their overall driving styles.


# Task
Create a scatter plot of the first two principal components, labeling each point with the driver's code to visualize clusters and relationships.

## visualize_pca

### Subtask:
Create a scatter plot of the first two principal components, labeling each point with the driver's code to visualize clusters and relationships.


## Summary:

### Data Analysis Key Findings
*   A scatter plot was generated using the first two principal components, with each point representing a driver and labeled by their unique driver code.
*   This visualization aims to reveal potential clusters or patterns among drivers based on their behavioral characteristics captured by the principal components.

### Insights or Next Steps
*   Examine the generated scatter plot for visual clusters of drivers to identify groups with similar driving behaviors.
*   Investigate the original features that contribute most to the identified principal components to understand the underlying meaning of these clusters.


In [ ]:
plt.figure(figsize=(12, 10))
sns.scatterplot(
    x='PC1', y='PC2',
    data=df_pca,
    hue=df_pca.index, # Color points by driver
    palette='tab20', # Use a distinct color palette
    s=150, # Size of the points
    alpha=0.8
)

# Add driver labels to the points
for i, driver in enumerate(df_pca.index):
    plt.annotate(driver, (df_pca['PC1'].iloc[i] + 0.5, df_pca['PC2'].iloc[i]), fontsize=9)

plt.title('PCA Visualization of Driver Driving Styles', fontsize=16)
plt.xlabel('Principal Component 1', fontsize=12)
plt.ylabel('Principal Component 2', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Driver')
plt.tight_layout()
plt.show()

# Task
Update the t-SNE calculation in cell `339b8e2a` to use the `telemetry` DataFrame instead of `df_standardized`.

## replace_df_standardized_with_telemetry

### Subtask:
Replace `df_standardized` with `telemetry` in the t-SNE calculation within cell `339b8e2a`.


## Summary:

### Data Analysis Key Findings
*   The `t-SNE` dimensionality reduction calculation was updated to utilize the `telemetry` DataFrame as its input, replacing the previously used `df_standardized` DataFrame. This signifies a shift in the data source for the visualization step.

### Insights or Next Steps
*   This modification indicates that the `telemetry` dataset is the intended focus for dimensionality reduction and subsequent clustering or pattern identification using `t-SNE`.
*   The next logical step would be to analyze the `t-SNE` output generated from the `telemetry` data to observe any emerging clusters or inherent structures within the dataset.
